# Formula 1 World Championship Analysis
## Exploratory Data Analysis
* **Goal:** Prepare and transform historical Formula 1 race data into features for a machine learning model that estimates the probability of a driver winning a race.

This notebook prepares the cleaned dataset for machine learning models.

In [108]:
# Import relevat libraries
import pandas as pd
import numpy as np

In [109]:
# Load CSV Files

circuits = pd.read_csv('../data/circuits.csv')
constructors = pd.read_csv('../data/constructors.csv')
driver_standings = pd.read_csv('../data/driver_standings.csv')
drivers = pd.read_csv('../data/drivers.csv')
qualifying = pd.read_csv('../data/qualifying.csv')
races = pd.read_csv('../data/races.csv')
results = pd.read_csv('../data/results.csv')

In [110]:
# Check results csv
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [111]:
# Check columns of results csv
results.columns

Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId'],
      dtype='object')

In [112]:
# Creating new feature - 'win'
results['win'] = (results['positionOrder'] == 1).astype(int)

# Check for output
print(results['win'].value_counts())

win
0    25631
1     1128
Name: count, dtype: int64


In [113]:
# Check the head with the new feature
print(results[['driverId', 'raceId', 'positionOrder', 'win']].head(10))

   driverId  raceId  positionOrder  win
0         1      18              1    1
1         2      18              2    0
2         3      18              3    0
3         4      18              4    0
4         5      18              5    0
5         6      18              6    0
6         7      18              7    0
7         8      18              8    0
8         9      18              9    0
9        10      18             10    0


In [114]:
# Merge with races.csv
results_fe = results.merge(
    races[['raceId', 'year', 'round', 'circuitId']],
    on='raceId',
    how='left'
)

# Check the new DataFrame
print(results_fe[['raceId', 'year', 'round', 'circuitId', 'win']].head(10))

   raceId  year  round  circuitId  win
0      18  2008      1          1    1
1      18  2008      1          1    0
2      18  2008      1          1    0
3      18  2008      1          1    0
4      18  2008      1          1    0
5      18  2008      1          1    0
6      18  2008      1          1    0
7      18  2008      1          1    0
8      18  2008      1          1    0
9      18  2008      1          1    0


In [115]:
# Sort races chronologically
results_fe = results_fe.sort_values(
    ['year', 'round', 'raceId']
).reset_index(drop=True)

In [116]:
# Check if it works
print(
    results_fe[
        ['raceId', 'year', 'round', 'driverId', 'win']
    ].head(20)
)

    raceId  year  round  driverId  win
0      833  1950      1       642    1
1      833  1950      1       786    0
2      833  1950      1       686    0
3      833  1950      1       704    0
4      833  1950      1       627    0
5      833  1950      1       619    0
6      833  1950      1       787    0
7      833  1950      1       741    0
8      833  1950      1       784    0
9      833  1950      1       778    0
10     833  1950      1       660    0
11     833  1950      1       579    0
12     833  1950      1       776    0
13     833  1950      1       669    0
14     833  1950      1       747    0
15     833  1950      1       785    0
16     833  1950      1       640    0
17     833  1950      1       589    0
18     833  1950      1       789    0
19     833  1950      1       661    0


In [117]:
# Count how many races each driver had participated in before the current race
results_fe['previous_races'] = (
    results_fe
    .groupby('driverId')
    .cumcount()
)

In [118]:
# Check the output with driverId 1
print(
    results_fe[
        results_fe['driverId'] == 1
    ][
        ['year', 'round', 'raceId', 'driverId', 'previous_races']
    ].head(10)
)

       year  round  raceId  driverId  previous_races
19243  2007      1      36         1               0
19264  2007      2      37         1               1
19286  2007      3      38         1               2
19308  2007      4      39         1               3
19330  2007      5      40         1               4
19351  2007      6      41         1               5
19373  2007      7      42         1               6
19397  2007      8      43         1               7
19419  2007      9      44         1               8
19447  2007     10      45         1               9


In [119]:
# Calculate the number of previous wins for each driver
results_fe['previous_wins'] = (
    results_fe
    .groupby('driverId')['win']
    .transform(lambda x: x.cumsum().shift(1, fill_value=0))
)

In [120]:
# Check the output
print(
    results_fe[
        ['year', 'round', 'raceId', 'driverId', 'win', 'previous_wins']
    ].head(20)
)

    year  round  raceId  driverId  win  previous_wins
0   1950      1     833       642    1              0
1   1950      1     833       786    0              0
2   1950      1     833       686    0              0
3   1950      1     833       704    0              0
4   1950      1     833       627    0              0
5   1950      1     833       619    0              0
6   1950      1     833       787    0              0
7   1950      1     833       741    0              0
8   1950      1     833       784    0              0
9   1950      1     833       778    0              0
10  1950      1     833       660    0              0
11  1950      1     833       579    0              0
12  1950      1     833       776    0              0
13  1950      1     833       669    0              0
14  1950      1     833       747    0              0
15  1950      1     833       785    0              0
16  1950      1     833       640    0              0
17  1950      1     833     

In [121]:
# Check the output for a winner
print(
    results_fe[
        results_fe['win'] == 1
    ][
        ['year', 'round', 'raceId', 'driverId', 'win', 'previous_wins']
    ].head(20)
)

     year  round  raceId  driverId  win  previous_wins
0    1950      1     833       642    1              0
23   1950      2     834       579    1              0
44   1950      3     835       593    1              0
79   1950      4     836       642    1              1
97   1950      5     837       579    1              1
111  1950      6     838       579    1              2
131  1950      7     839       642    1              2
160  1951      1     825       579    1              3
181  1951      2     826       766    1              0
215  1951      3     827       642    1              3
228  1951      4     828       786    1              0
251  1951      4     828       579    1              4
254  1951      5     829       498    1              0
274  1951      6     830       647    1              0
296  1951      7     831       647    1              1
319  1951      8     832       579    1              5
339  1952      1     817       641    1              0
361  1952 

In [122]:
# Calculate the number of previous points for each driver
results_fe['previous_points'] = (
    results_fe
    .groupby('driverId')['points']
    .transform(lambda x: x.cumsum().shift(1, fill_value=0))
)

In [123]:
# Check the output
print(
    results_fe[
        ['year', 'round', 'raceId', 'driverId', 'points', 'previous_points']
    ].head(20)
)

    year  round  raceId  driverId  points  previous_points
0   1950      1     833       642     9.0              0.0
1   1950      1     833       786     6.0              0.0
2   1950      1     833       686     4.0              0.0
3   1950      1     833       704     3.0              0.0
4   1950      1     833       627     2.0              0.0
5   1950      1     833       619     0.0              0.0
6   1950      1     833       787     0.0              0.0
7   1950      1     833       741     0.0              0.0
8   1950      1     833       784     0.0              0.0
9   1950      1     833       778     0.0              0.0
10  1950      1     833       660     0.0              0.0
11  1950      1     833       579     0.0              0.0
12  1950      1     833       776     0.0              0.0
13  1950      1     833       669     0.0              0.0
14  1950      1     833       747     0.0              0.0
15  1950      1     833       785     0.0              0

In [124]:
# Calculate the avg of previous position
results_fe['previous_avg_position'] = (
    results_fe
    .groupby('driverId')['positionOrder']
    .transform(lambda x: x.expanding().mean().shift(1))
)

In [125]:
# Check the output
print(
    results_fe[
        ['year', 'round', 'raceId', 'driverId', 'positionOrder', 'previous_avg_position']
    ].head(20)
)

    year  round  raceId  driverId  positionOrder  previous_avg_position
0   1950      1     833       642              1                    NaN
1   1950      1     833       786              2                    NaN
2   1950      1     833       686              3                    NaN
3   1950      1     833       704              4                    NaN
4   1950      1     833       627              5                    NaN
5   1950      1     833       619              6                    NaN
6   1950      1     833       787              7                    NaN
7   1950      1     833       741              8                    NaN
8   1950      1     833       784              9                    NaN
9   1950      1     833       778             10                    NaN
10  1950      1     833       660             11                    NaN
11  1950      1     833       579             12                    NaN
12  1950      1     833       776             13                

In [126]:
# Check previous avg position with explicit driverId 1
print(
    results_fe[
        results_fe['driverId'] == 1
    ][
        ['year', 'round', 'raceId', 'driverId', 'positionOrder', 'previous_avg_position']
    ].head(10)
)

       year  round  raceId  driverId  positionOrder  previous_avg_position
19243  2007      1      36         1              3                    NaN
19264  2007      2      37         1              2               3.000000
19286  2007      3      38         1              2               2.500000
19308  2007      4      39         1              2               2.333333
19330  2007      5      40         1              2               2.250000
19351  2007      6      41         1              1               2.200000
19373  2007      7      42         1              1               2.000000
19397  2007      8      43         1              3               1.857143
19419  2007      9      44         1              3               2.000000
19447  2007     10      45         1              9               2.111111


In [127]:
# Identify previous podium finishes
results_fe['podium'] = (
    results_fe['positionOrder'] <= 3
).astype(int)

In [128]:
# Calculate the number of previous podiums
results_fe['previous_podiums'] = (
    results_fe
    .groupby('driverId')['podium']
    .transform(lambda x: x.cumsum().shift(1, fill_value=0))
)

In [129]:
# Check the output
print(
    results_fe[
        ['year', 'round', 'raceId', 'driverId', 'positionOrder', 'podium', 'previous_podiums']
    ].head(20)
)

    year  round  raceId  driverId  positionOrder  podium  previous_podiums
0   1950      1     833       642              1       1                 0
1   1950      1     833       786              2       1                 0
2   1950      1     833       686              3       1                 0
3   1950      1     833       704              4       0                 0
4   1950      1     833       627              5       0                 0
5   1950      1     833       619              6       0                 0
6   1950      1     833       787              7       0                 0
7   1950      1     833       741              8       0                 0
8   1950      1     833       784              9       0                 0
9   1950      1     833       778             10       0                 0
10  1950      1     833       660             11       0                 0
11  1950      1     833       579             12       0                 0
12  1950      1     833  

In [130]:
# Check all columns
results_fe.columns

Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId', 'win', 'year', 'round', 'circuitId',
       'previous_races', 'previous_wins', 'previous_points',
       'previous_avg_position', 'podium', 'previous_podiums'],
      dtype='object')

In [131]:
# New feature - constructor_wins
results_fe['constructor_wins'] = (
    results_fe['positionOrder'] == 1
).astype(int)

In [132]:
# Identify constructor wins per race
constructor_race_wins = (
    results_fe[results_fe['positionOrder'] ==1]
    [['raceId', 'year', 'round', 'constructorId']]
    .drop_duplicates()
)

In [133]:
# Count previous wins for each constructor
constructor_race_wins['previous_constructor_wins'] = (
    constructor_race_wins
    .groupby('constructorId')
    .cumcount()
)

In [134]:
# Merge previous constructor wins with the main dataset
results_fe = results_fe.merge(
    constructor_race_wins[
        ['raceId', 'constructorId', 'previous_constructor_wins']
    ],
    on=['raceId', 'constructorId'],
    how='left'
)

In [135]:
print(
    results_fe[
    ['year', 'round', 'raceId',
         'driverId', 'constructorId',
         'positionOrder',
         'previous_constructor_wins']
    ].head(30)
)

    year  round  raceId  driverId  constructorId  positionOrder  \
0   1950      1     833       642             51              1   
1   1950      1     833       786             51              2   
2   1950      1     833       686             51              3   
3   1950      1     833       704            154              4   
4   1950      1     833       627            154              5   
5   1950      1     833       619            151              6   
6   1950      1     833       787            151              7   
7   1950      1     833       741            154              8   
8   1950      1     833       784            105              9   
9   1950      1     833       778            105             10   
10  1950      1     833       660            154             11   
11  1950      1     833       579             51             12   
12  1950      1     833       776            126             13   
13  1950      1     833       669            105             1

In [136]:
# Filling NaN with 0 in previous_constructor_wins
results_fe['previous_constructor_wins'] = (
    results_fe['previous_constructor_wins']
    .fillna(0)
)

In [137]:
print(
    results_fe[
        ['year', 'round', 'raceId',
         'driverId', 'constructorId',
         'positionOrder',
         'previous_constructor_wins']
    ].head(30)
)

    year  round  raceId  driverId  constructorId  positionOrder  \
0   1950      1     833       642             51              1   
1   1950      1     833       786             51              2   
2   1950      1     833       686             51              3   
3   1950      1     833       704            154              4   
4   1950      1     833       627            154              5   
5   1950      1     833       619            151              6   
6   1950      1     833       787            151              7   
7   1950      1     833       741            154              8   
8   1950      1     833       784            105              9   
9   1950      1     833       778            105             10   
10  1950      1     833       660            154             11   
11  1950      1     833       579             51             12   
12  1950      1     833       776            126             13   
13  1950      1     833       669            105             1

In [138]:
# Create new feature - previous_constructor_points
constructor_race_points = (
    results_fe
    .groupby(['raceId', 'year', 'round', 'constructorId'])['points']
    .sum()
    .reset_index()
)

In [139]:
# Sort values
constructor_race_points = constructor_race_points.sort_values(
    ['constructorId', 'year', 'round']
)

In [140]:
# Check the output
print(
    constructor_race_points
)

       raceId  year  round  constructorId  points
8761      674  1968      8              1     0.0
8794      677  1968     11              1     0.0
8387      632  1971      1              1     1.0
8397      633  1971      2              1     2.0
8407      634  1971      3              1     3.0
...       ...   ...    ...            ...     ...
12987    1140  2024     20            215     0.0
12997    1141  2024     21            215     8.0
13007    1142  2024     22            215     2.0
13017    1143  2024     23            215     0.0
13027    1144  2024     24            215     0.0

[13028 rows x 5 columns]


In [141]:
# Calculate the previous points
constructor_race_points['previous_constructor_points'] = (
    constructor_race_points
    .groupby('constructorId')['points']
    .transform(lambda x: x.cumsum().shift(1, fill_value=0))
)

In [142]:
# Merge with the main dataset
results_fe = results_fe.merge(
    constructor_race_points[
        ['raceId', 'constructorId', 'previous_constructor_points']
    ],
    on=['raceId', 'constructorId'],
    how='left'
)

In [143]:
# Check the dataset
print(
    results_fe[
        ['year', 'round', 'raceId',
         'driverId', 'constructorId',
         'points',
         'previous_constructor_points']
    ].head(30)
)

    year  round  raceId  driverId  constructorId  points  \
0   1950      1     833       642             51     9.0   
1   1950      1     833       786             51     6.0   
2   1950      1     833       686             51     4.0   
3   1950      1     833       704            154     3.0   
4   1950      1     833       627            154     2.0   
5   1950      1     833       619            151     0.0   
6   1950      1     833       787            151     0.0   
7   1950      1     833       741            154     0.0   
8   1950      1     833       784            105     0.0   
9   1950      1     833       778            105     0.0   
10  1950      1     833       660            154     0.0   
11  1950      1     833       579             51     0.0   
12  1950      1     833       776            126     0.0   
13  1950      1     833       669            105     0.0   
14  1950      1     833       747            105     0.0   
15  1950      1     833       785       